# WM-811K 취약 클래스 개선 후보 비교

Scratch, Loc, Edge-Loc 개선 후보 3개를 train으로 학습하고 validation으로만 비교합니다. Test 추론은 수행하지 않습니다.

In [ ]:
!pip install -q pandas==2.2.3 scikit-learn==1.6.1 matplotlib==3.10.0
from google.colab import drive, files
from pathlib import Path
import shutil, subprocess, sys, torch
drive.mount('/content/drive')
assert torch.cuda.is_available(), '런타임을 T4 GPU로 변경한 뒤 다시 실행하세요.'
print('GPU:', torch.cuda.get_device_name(0))

아래 셀에서 기본 학습 스크립트, 개선 후보 스크립트, split_assignments.csv를 한 개씩 선택합니다.

In [ ]:
print('1/3: train_wm811k_cnn.py를 선택하세요.')
base_upload = files.upload()
assert len(base_upload) == 1, '파일 한 개만 선택하세요.'
Path('/content/train_wm811k_cnn.py').write_bytes(next(iter(base_upload.values())))
print('2/3: train_wm811k_weak_class_candidates.py를 선택하세요.')
candidate_upload = files.upload()
assert len(candidate_upload) == 1, '파일 한 개만 선택하세요.'
Path('/content/train_wm811k_weak_class_candidates.py').write_bytes(next(iter(candidate_upload.values())))
print('3/3: split_assignments.csv를 선택하세요.')
split_upload = files.upload()
assert len(split_upload) == 1, '파일 한 개만 선택하세요.'
Path('/content/split_assignments.csv').write_bytes(next(iter(split_upload.values())))
print('업로드 완료')

In [ ]:
DATA_DIR = Path('/content/drive/MyDrive/AI_project/WM811K_processed')
MAPS = DATA_DIR / 'wafer_maps_64.npy'
LABELS = DATA_DIR / 'labels.npy'
ASSIGNMENTS = Path('/content/split_assignments.csv')
SCRIPT = Path('/content/train_wm811k_weak_class_candidates.py')
for path in (MAPS, LABELS, ASSIGNMENTS, SCRIPT, Path('/content/train_wm811k_cnn.py')):
    assert path.is_file(), f'필수 파일이 없습니다: {path}'
OUTPUT_DIR = Path('/content/drive/MyDrive/AI_project/WM811K_weak_class_experiment/validation_only_seed42')
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
command = [sys.executable, str(SCRIPT), '--maps', str(MAPS), '--labels', str(LABELS), '--assignments', str(ASSIGNMENTS), '--output-dir', str(OUTPUT_DIR), '--epochs', '20', '--batch-size', '256', '--num-workers', '2', '--hard-negative-fraction', '0.10', '--seed', '42']
process = subprocess.Popen(command, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
for line in process.stdout:
    print(line, end='')
return_code = process.wait()
assert return_code == 0, f'학습 스크립트 실패: 종료 코드 {return_code}'
print('Validation-only 후보 비교 완료:', OUTPUT_DIR)

In [ ]:
ZIP_BASE = Path('/content/wm811k_weak_class_validation_results')
zip_path = Path(shutil.make_archive(str(ZIP_BASE), 'zip', root_dir=OUTPUT_DIR))
print('다운로드 파일:', zip_path, f'{zip_path.stat().st_size / 1024**2:.2f} MB')
files.download(str(zip_path))

다운로드한 ZIP을 결과물/wm811k/colab_가져오기 폴더에 넣습니다. 선택 후보도 아직 배포하지 않으며, 여러 seed 재현 후에만 최종 test 평가를 검토합니다.